# **RoboPianist drum performance tutorial**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/google-research/robopianist/blob/main/drum_tutorial.ipynb)


> <p><small><small>Copyright 2024 The RoboPianist Authors.</small></p>
> <p><small><small>Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at <a href="http://www.apache.org/licenses/LICENSE-2.0">http://www.apache.org/licenses/LICENSE-2.0</a>.</small></small></p>
> <p><small><small>Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.</small></small></p>


# Installing RoboPianist and drum demo dependencies


In [ ]:
# @title Install dependencies (Colab users: enable a GPU runtime first)
from IPython.display import clear_output

print("Installing packages...")
%pip install -q robopianist>=1.0.6 dm-control>=1.0.16 imageio[ffmpeg]
%env MUJOCO_GL=egl

clear_output()
print("Dependencies installed. If you are running on Colab, remember to enable a GPU runtime.")


# Imports


In [ ]:
# @title Imports used throughout the notebook
from base64 import b64encode
from pathlib import Path

import imageio.v2 as imageio
import numpy as np
from IPython.display import HTML, display

from dm_control import mjcf
from robopianist.models.drum import drum_mjcf


# Helper functions


In [ ]:
# @title Utility helpers for visualization and simulation

VIDEO_DIR = Path('videos')
VIDEO_DIR.mkdir(exist_ok=True)

def play_video(filename: str) -> None:
    """Embed an MP4 video directly inside the notebook."""
    path = Path(filename)
    with path.open('rb') as f:
        video = f.read()
    b64 = b64encode(video).decode('utf-8')
    display(
        HTML(
            f"""
            <video width="720" controls loop>
              <source src="data:video/mp4;base64,{b64}" type="video/mp4">
            </video>
            """
        )
    )

def interpolate_controls(times: np.ndarray, values: np.ndarray, t: float) -> np.ndarray:
    """Linearly interpolate joint targets at time ``t``."""
    return np.array([np.interp(t, times, values[:, i]) for i in range(values.shape[1])])

def simulate_and_render(
    physics: mjcf.Physics,
    times: np.ndarray,
    values: np.ndarray,
    *,
    camera_id: str = 'front',
    fps: int = 30,
    hold_steps: int = 45,
    resolution: tuple[int, int] = (480, 720),
) -> list[np.ndarray]:
    """Run the control sequence and return RGB frames."""
    frames: list[np.ndarray] = []
    dt = physics.timestep()
    steps_per_frame = max(1, int(round((1.0 / fps) / dt)))
    total_steps = int(np.ceil(times[-1] / dt))

    for step in range(total_steps):
        t = step * dt
        ctrl = interpolate_controls(times, values, t)
        physics.control[:] = ctrl
        physics.step()
        if step % steps_per_frame == 0:
            frame = physics.render(height=resolution[0], width=resolution[1], camera_id=camera_id)
            frames.append(frame)

    # Hold the final pose for a short beat.
    for _ in range(hold_steps):
        physics.control[:] = values[-1]
        physics.step()
        frame = physics.render(height=resolution[0], width=resolution[1], camera_id=camera_id)
        frames.append(frame)

    return frames


# Drum kit with a simple robotic arm


In [ ]:
# @title Build a procedural drum kit with a 4-DoF striking arm

# Start from the procedural drum kit provided by robopianist.
drum_model = drum_mjcf.build(add_actuators=False)
drum_model.option.timestep = 0.002

world = drum_model.worldbody

# Mounting post that anchors the arm behind the snare drum.
arm_mount = world.add('body', name='arm_mount', pos=[-0.2, -0.55, 0.35])
arm_mount.add(
    'geom',
    name='arm_post',
    type='capsule',
    fromto=[0, 0, -0.35, 0, 0, 0.05],
    size=[0.06],
    rgba=[0.2, 0.2, 0.2, 1.0],
)

shoulder_body = arm_mount.add('body', name='shoulder_body', pos=[0, 0, 0.05])
shoulder_yaw = shoulder_body.add(
    'joint',
    name='shoulder_yaw',
    type='hinge',
    axis=[0, 0, 1],
    limited=True,
    range=[-1.2, 1.2],
    damping=3.0,
)
shoulder_body.add(
    'geom',
    name='shoulder_base',
    type='capsule',
    fromto=[0, 0, 0, 0, 0, 0.2],
    size=[0.05],
    rgba=[0.3, 0.3, 0.35, 1.0],
)

upper_arm = shoulder_body.add('body', name='upper_arm', pos=[0, 0, 0.2])
shoulder_pitch = upper_arm.add(
    'joint',
    name='shoulder_pitch',
    type='hinge',
    axis=[0, 1, 0],
    limited=True,
    range=[-1.4, 1.2],
    damping=2.0,
)
upper_arm.add(
    'geom',
    name='upper_arm_geom',
    type='capsule',
    fromto=[0, 0, 0, 0, 0, 0.45],
    size=[0.04],
    rgba=[0.4, 0.4, 0.45, 1.0],
)

forearm = upper_arm.add('body', name='forearm', pos=[0, 0, 0.45])
elbow_pitch = forearm.add(
    'joint',
    name='elbow_pitch',
    type='hinge',
    axis=[0, 1, 0],
    limited=True,
    range=[-2.0, 0.5],
    damping=1.5,
)
forearm.add(
    'geom',
    name='forearm_geom',
    type='capsule',
    fromto=[0, 0, 0, 0, 0, 0.35],
    size=[0.035],
    rgba=[0.5, 0.5, 0.55, 1.0],
)

wrist = forearm.add('body', name='wrist', pos=[0, 0, 0.35])
wrist_yaw = wrist.add(
    'joint',
    name='wrist_yaw',
    type='hinge',
    axis=[0, 0, 1],
    limited=True,
    range=[-1.0, 1.0],
    damping=0.5,
)
wrist.add(
    'geom',
    name='wrist_geom',
    type='capsule',
    fromto=[0, 0, 0, 0, 0, 0.2],
    size=[0.025],
    rgba=[0.45, 0.45, 0.5, 1.0],
)

stick = wrist.add('body', name='drum_stick', pos=[0, 0, 0.2])
stick.add(
    'geom',
    name='drum_stick_geom',
    type='capsule',
    fromto=[0, 0, 0, 0.35, 0, 0],
    size=[0.01],
    rgba=[0.8, 0.6, 0.3, 1.0],
)
stick.add('site', name='stick_tip', pos=[0.35, 0, 0], size=[0.01], rgba=[1, 0, 0, 1.0])

control_joints = [shoulder_yaw, shoulder_pitch, elbow_pitch, wrist_yaw]
control_joint_names = [joint.attrib['name'] for joint in control_joints]

for joint in control_joints:
    drum_model.actuator.add(
        'position',
        name=f"{joint.attrib['name']}_actuator",
        joint=joint,
        ctrlrange=list(joint.range),
        kp=200,
    )

physics = mjcf.Physics.from_mjcf_model(drum_model)
initial_pose = np.array([0.45, -0.25, 0.75, 0.25])
for joint_name, value in zip(control_joint_names, initial_pose):
    physics.named.data.qpos[joint_name] = value
physics.forward()

print('Controlled joints:', control_joint_names)
print('Number of actuators:', physics.model.nu)
print('Available cameras:', [name.decode('utf-8') for name in physics.model.cam_names])


# Define a striking motion


In [ ]:
# @title Keyframe targets for a snare hit

keyframe_times = np.array([0.0, 0.45, 0.65, 0.9, 1.3])
keyframe_values = np.array(
    [
        [0.45, -0.25, 0.75, 0.25],  # ready pose
        [0.25, -0.15, 0.4, 0.2],    # wind-up
        [0.05, -0.7, -1.05, 0.0],   # downward strike
        [0.35, -0.3, 0.55, 0.15],   # rebound
        [0.45, -0.25, 0.75, 0.25],  # return to ready
    ]
)

print('Keyframe times (s):', keyframe_times)
print('Target joint angles (rad):')
for name, column in zip(control_joint_names, keyframe_values.T):
    print(f"  {name:>15s}: {np.array2string(column, precision=2)}")


# Simulate and render the drum performance


In [ ]:
# @title Run the trajectory and export a video

physics = mjcf.Physics.from_mjcf_model(drum_model)
for joint_name, value in zip(control_joint_names, keyframe_values[0]):
    physics.named.data.qpos[joint_name] = value
physics.forward()

frames = simulate_and_render(
    physics,
    times=keyframe_times,
    values=keyframe_values,
    camera_id='front',
    fps=30,
    hold_steps=60,
    resolution=(480, 720),
)

output_path = VIDEO_DIR / 'drum_snare_hit.mp4'
imageio.mimsave(output_path, frames, fps=30, macro_block_size=None)

print(f'Saved video to {output_path}')
play_video(output_path)
